In [1]:
import torch
import torch.nn as nn

In [156]:
# MES loss function implementations

In [143]:
def mse_loss(preds, labels):
    # dim (batch_size, seq_len)
    return ((preds - labels)**2).mean()

In [149]:
preds = torch.rand(num_sample, 25, requires_grad=True)
labels = torch.rand(num_sample, 25, requires_grad=True)

In [150]:
loss = mse_loss(preds, labels)

In [151]:
loss

tensor(0.1688, grad_fn=<MeanBackward0>)

In [153]:
mse_loss2 = nn.MSELoss()

In [154]:
loss_2 = mse_loss(preds, labels)

In [155]:
loss_2

tensor(0.1688, grad_fn=<MeanBackward0>)

In [157]:
# Masked CE

In [167]:
logits = torch.rand(num_sample, 25, 48)
targets = torch.randint(0, 48, (num_sample, 25))
pad_id = 0

In [168]:
targets.size()

torch.Size([500, 25])

In [169]:
targets[:,20:] = 0

In [174]:
targets[0, -1] == torch.tensor(0.0).float()

tensor(True)

In [210]:
mask = (targets.view(-1) != pad_id)

In [212]:
mask[:25]

tensor([ True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
        False, False, False, False, False])

In [175]:
ce_loss = nn.CrossEntropyLoss()

In [209]:
# The high dimension take-ins for cross_entorpy has dimension comes later, easiest to view(-1)
# logits : (N, C) or (N, C, d1, d2, ...), where C is the class dimension, which is 48 in this case
# targets: (N)     or (N, d1, d2, ...)
loss_1 = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))

In [203]:
loss_2 = ce_loss(logits.view(-1, logits.size(-1)), targets.view(-1))

In [216]:
def masked_cross_entropy(logits, targets, pad_id):
    loss = F.cross_entropy(
        logits.view(-1, logits.size(-1)),
        targets.view(-1),
        reduction="none")
    mask = (targets.view(-1) != pad_id).float()
    output = (loss*mask).sum() / mask.sum()
    return output

In [217]:
masked_loss = masked_cross_entropy(logits, targets, pad_id)

In [218]:
masked_loss

tensor(3.9139)

In [222]:
# contrastive loss for embedding pairs
# L=y⋅d**2+(1−y)⋅max(0,m−d)**2
# d: Euclidian distance
# y -> label. y==1 positive pair, y==0 negative pair

In [235]:
z1 = torch.rand(num_sample, 48, requires_grad=True)
z2 = torch.rand(num_sample, 48, requires_grad=True)
y = torch.randint(0, 2, (num_sample,))

In [253]:
def contrastive_loss(z1, z2, label, margin=1.0):
    dist = torch.norm(z1 - z2, dim=-1)
    pos = (y * dist)**2
    neg = (1-y) * torch.clamp(margin - dist, min=0)**2
    return (pos+neg).mean()

In [250]:
def contrastive_loss_2(z1, z2, label, margin=1.0):
    dist = torch.norm(z1 - z2, dim=-1)
    pos = (y * dist)**2
    m_d = torch.maximum(margin-dist, torch.zeros_like(dist))
    neg = (1-y) * m_d**2
    return (pos+neg).mean()

In [254]:
loss = contrastive_loss(z1, z2, y)

In [255]:
loss

tensor(4.0269, grad_fn=<MeanBackward0>)

In [256]:
# Implement knowledge distillation loss using KL divergence between teacher and student logits.

In [268]:
import torch.nn.functional as F

def distillation_loss(student_logits, teacher_logits, T=2.0):
    p_student = F.log_softmax(student_logits / T, dim=-1)
    p_teacher = F.softmax(teacher_logits / T, dim=-1)
    return F.kl_div(p_student, p_teacher, reduction="batchmean") * (T * T)

In [ ]:
# in langurage model with mask
def distillation_loss_lm(student_logits, teacher_logits, attention_mask, T=2.0):
    p_student = F.log_softmax(student_logits / T, dim=-1)
    p_teacher = F.softmax(teacher_logits / T, dim=-1)

    kl = F.kl_div(p_student, p_teacher, reduction="none")  # (B, T, V)
    kl = kl.sum(dim=-1)  # sum over vocab -> (B, T)

    kl = kl * attention_mask  # mask padding
    return kl.sum() / attention_mask.sum() * (T * T)


In [269]:
# dim (batch_size, num_classes)
student_logits = torch.rand(num_sample, 48, requires_grad = True)
teacher_logits = torch.rand(num_sample, 48, requires_grad = True)

In [270]:
loss = distillation_loss(student_logits, teacher_logits)

In [271]:
loss

tensor(0.0801, grad_fn=<MulBackward0>)

In [272]:
# Implement the pairwise reward model loss used in RLHF.

In [273]:
# neg-log likelihood on the prob that the preferred answer (i.e., r_pos) wins
def preference_loss(r_pos, r_neg):
    return -torch.log(torch.sigmoid(r_pos - r_neg)).mean()

In [ ]:
# for stability you do softplus, which is same but more stable than log(sigmoid)
def preference_loss(r_pos, r_neg):
    return F.softplus(-(r_pos - r_neg)).mean()

In [275]:
r_pos_logits = torch.rand(num_sample, requires_grad=True)
r_neg_logits = torch.rand(num_sample, requires_grad=True)

In [276]:
loss = preference_loss(r_pos_logits, r_neg_logits)

In [277]:
loss

tensor(0.7000, grad_fn=<NegBackward0>)

In [ ]:
# sigmoid is the softmax for 2 classes (let's test it)

In [274]:
# You have a regression task and a classification task.
# Implement a weighted multi-task loss.

In [282]:
def multitask_loss(y_reg, y_reg_hat, y_cls, y_cls_hat, alpha=0.5):
    reg_loss = F.mse_loss(y_reg_hat, y_reg)
    cls_loss = F.cross_entropy(y_cls_hat, y_cls)
    return alpha * reg_loss + (1 - alpha) * cls_loss

In [ ]:
# data is wrong here, review later

In [288]:
y_reg = torch.rand(num_sample, requires_grad=True)
y_reg_hat = torch.rand(num_sample, requires_grad=True)
y_cls = torch.randint(0, 2, (num_sample,))
y_cls_hat = torch.rand(num_sample, 2, requires_grad=True)

In [289]:
loss = multitask_loss(y_reg, y_reg_hat, y_cls, y_cls_hat)

In [290]:
loss

tensor(0.4423, grad_fn=<AddBackward0>)

In [287]:
# Implement a loss function with a custom backward.

In [300]:
class CustomLossFn(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        ctx.save_for_backward(x)
        return x.pow(2).mean()

    @staticmethod
    def backward(ctx, grad_output):
        (x,) = ctx.saved_tensors
        return grad_output * 2 * x / x.numel()

In [301]:
# Create input tensor
x = torch.tensor([1.0, -2.0, 3.0], requires_grad=True)

# Call the custom loss
loss = CustomLossFn.apply(x)

print("loss:", loss.item())


loss: 4.666666507720947


In [302]:
loss.backward()

In [303]:
x.grad

tensor([ 0.6667, -1.3333,  2.0000])

In [304]:
# Wrap it in an nn.Module so it can be called like pre-built loss classes
class CustomLoss(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return CustomLossFn.apply(x)


In [305]:
loss_fn = CustomLoss()

x = torch.tensor([1.0, -2.0, 3.0], requires_grad=True)
loss = loss_fn(x)

loss.backward()
print(loss.item(), x.grad)


4.666666507720947 tensor([ 0.6667, -1.3333,  2.0000])


In [306]:
# Wrap it like a nn.Functional loss function
def custom_loss(x):
    return CustomLossFn.apply(x)

In [307]:
loss = custom_loss(x)
loss

tensor(4.6667, grad_fn=<CustomLossFnBackward>)

In [ ]:
# Implement binary cross-entropy without numerical overflow.

In [ ]:
def stable_bce(logits, targets):
    return torch.mean(
        torch.clamp(logits, min=0)
        - logits * targets
        + torch.log1p(torch.exp(-torch.abs(logits)))
    )